# Proyecto Final: Chatbot de Atención al Cliente para Ripley Chile


Antes de ejecutar el código, asegúrate de tener configuradas las siguientes variables de entorno:

```bash
export GITHUB_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aqui"
```

## Instalación de dependencias

Para ejecutar este proyecto se requieren las bibliotecas `openai`, `langchain` y `langchain-openai`.

Ejecutar solo si es necesario
```bash
pip install langchain langchain-openai streamlit langchain-community langchain-text-splitters faiss-cpu
```

## Integración de técnicas avanzadas en el chatbot

- **Prompt Chaining:** Se estructura el flujo de conversación utilizando el historial (`history.messages`), permitiendo que cada respuesta considere el contexto previo del usuario.

- **Meta-Prompting:** Se define un `SystemMessage` claro y específico que guía el comportamiento del modelo (rol, tono, tipo de respuestas), optimizando la generación de respuestas.

- **Zero-Shot Prompting:** Se entregan instrucciones claras al modelo sin necesidad de ejemplos en la clasificación inicial, permitiendo que el modelo infiera la intención directamente desde el mensaje del usuario.

- **Few-Shot Prompting:** Se incorporan ejemplos de entrada y salida dentro del prompt de clasificación para mejorar la precisión del modelo en la identificación de intenciones.

- **Chain of Thought (CoT) Implícito:** Se instruye al modelo a “pensar paso a paso” antes de responder, sin mostrar el razonamiento, mejorando la calidad de la respuesta final sin aumentar el ruido en la salida.


In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os
import time

# =========================
# CONFIGURACION DEL MODELO
# =========================
# Se configura el modelo de lenguaje que se utilizara
# Se obtienen credenciales desde variables de entorno
# Se define el modelo, temperatura, streaming y limite de tokens
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    streaming=True,
    temperature=0.3,
    max_tokens=250
)

# =========================
# MEMORIA POR SESION
# =========================
# Diccionario para almacenar el historial de conversaciones por sesion
store = {}

# Funcion que obtiene o crea el historial de una sesion
def get_session_history(session_id: str):
    if session_id not in store:
        # Si no existe la sesion, se crea una nueva en memoria
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# =========================
# 🔥 NUEVO: BASE DE CONOCIMIENTO (RAG)
# =========================
documents = [
    "Las devoluciones en Ripley se pueden realizar dentro de 10 días con boleta y producto en buen estado.",
    "Los despachos demoran entre 2 y 5 días hábiles dependiendo de la región.",
    "Las compras online pueden ser retiradas en tienda sin costo adicional.",
    "Los productos cuentan con garantía legal de 6 meses.",
    "Puedes pagar con tarjeta Ripley, tarjetas de crédito y débito.",
    "Para seguimiento de pedidos debes ingresar a tu cuenta en ripley.cl.",
    "Los cambios de productos están sujetos a disponibilidad de stock."
]

# =========================
# 🔥 NUEVO: CHUNKING
# =========================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))

# =========================
# 🔥 NUEVO: EMBEDDINGS + FAISS
# =========================
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.getenv("OPENAI_BASE_URL")
)

# Crear base vectorial
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)

# Crear retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# =========================
# PROMPT CHAINING: CLASIFICAR INTENCION
# =========================
# Funcion que usa el modelo para clasificar la intencion del usuario
def clasificar_intencion(mensaje):
    # Se construye el prompt con instrucciones y ejemplos (few-shot)
    prompt = f"""
    Clasifica la intencion del usuario en UNA sola palabra:
    - compra
    - reclamo
    - seguimiento
    - informacion

    Ejemplos:
    Mensaje: Quiero devolver un producto
    Respuesta: reclamo

    Mensaje: Donde esta mi pedido?
    Respuesta: seguimiento

    Mensaje: Quiero comprar un notebook
    Respuesta: compra

    Instrucciones:
    - Analiza cuidadosamente el mensaje
    - Piensa paso a paso antes de responder (no muestres tu razonamiento)
    - Responde SOLO con una palabra en minuscula

    Mensaje: {mensaje}
    Respuesta:
    """
    try:
        # Se envia el prompt al modelo y se obtiene la respuesta
        return llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    except:
        # En caso de error se retorna una intencion por defecto
        return "informacion"

# =========================
# META-PROMPTING: SYSTEM DINAMICO
# =========================
# Funcion que define el rol del chatbot segun la intencion detectada
def obtener_system_prompt(intencion):
    if "reclamo" in intencion:
        # Caso reclamo: respuesta empatica y solucion
        return (
            "Eres un asistente de atencion al cliente de Ripley Chile. "
            "El usuario tiene un reclamo. Responde con empatia, pide disculpas "
            "y ofrece una solucion clara y concreta."
        )
    elif "compra" in intencion:
        # Caso compra: rol de vendedor
        return (
            "Eres un asistente de ventas de Ripley Chile. "
            "Ayuda al usuario a elegir productos, recomienda opciones "
            "y responde dudas de compra de forma clara y util."
        )
    elif "seguimiento" in intencion:
        # Caso seguimiento: informacion de pedidos
        return (
            "Eres un asistente de seguimiento de pedidos de Ripley Chile. "
            "Entrega informacion clara sobre estados de envio, tiempos y procesos."
        )
    else:
        # Caso general: atencion al cliente
        return (
            "Eres un asistente virtual de atencion al cliente de Ripley Chile. "
            "Ayudas con compras, productos, despachos, devoluciones, cambios y pagos. "
            "Respondes de forma clara, amable y breve. "
            "Si no sabes algo con certeza, indicalo sin inventar informacion."
        )

# =========================
# CHATBOT PRINCIPAL
# =========================
# Funcion principal que ejecuta el chatbot
def chatbot_ripley():
    print("=== CHATBOT RIPLEY CHILE ===")
    print("Escribe 'salir' para terminar la conversacion.\n")
    
    # Se define un id de sesion para mantener el historial
    session_id = "chat_ripley"

    while True:
        # Se recibe la entrada del usuario
        user_input = input("🧑 Tú: ")

        # Condicion para salir del chatbot
        if user_input.lower() in ["salir", "exit", "quit"]:
            print("\n👋 Gracias por usar el chatbot de Ripley!")
            break

        # Si el input esta vacio, se ignora
        if not user_input.strip():
            continue

        print("\n🤖 RipleyBot: ", end="", flush=True)

        try:
            # Se obtiene el historial de la sesion
            history = get_session_history(session_id)

            # PROMPT CHAINING: detectar intencion
            intencion = clasificar_intencion(user_input)
            print(f"\n🔍 Intencion detectada: {intencion}")

            # 🔥 MODIFICADO: RAG con FAISS
            relevant_docs = retriever.invoke(user_input)

            if relevant_docs:
                context = "\n".join([doc.page_content for doc in relevant_docs])
            else:
                context = "No hay informacion relevante."

            # META-PROMPTING: system dinamico
            system_prompt = obtener_system_prompt(intencion)

            # 🔥 MODIFICADO: prompt con contexto
            rag_prompt = f"""
            Contexto relevante:
            {context}
            
            Historial de la conversación:
            {history.messages}
            
            Pregunta del usuario:
            {user_input}
            
            Instrucciones:
            - Usa el contexto si es útil
            - Si la respuesta está en el historial, úsalo
            - Si no sabes, dilo claramente
            """

            # Se construye el mensaje completo con:
            # system prompt + historial + nuevo mensaje
            messages = [
                SystemMessage(content=system_prompt)
            ] + history.messages + [HumanMessage(content=rag_prompt)]

            full_response = ""

            # STREAMING: respuesta en tiempo real
            for chunk in llm.stream(messages):
                content = chunk.content
                print(content, end="", flush=True)
                full_response += content
                time.sleep(0.01)

            print()

            # Guardar conversacion en memoria
            history.add_user_message(user_input)
            history.add_ai_message(full_response)

        except KeyboardInterrupt:
            # Manejo de interrupcion manual
            print("\n\n⏸️ Conversacion interrumpida")
            break
        except Exception as e:
            # Manejo de errores generales
            print(f"\n❌ Error: {e}")

if __name__ == "__main__":
    chatbot_ripley()

=== CHATBOT RIPLEY CHILE ===
Escribe 'salir' para terminar la conversacion.


🤖 RipleyBot: 
🔍 Intencion detectada: informacion
¡Hola, Sebastian! ¿En qué puedo ayudarte hoy? Si tienes alguna consulta sobre compras, productos, despachos, devoluciones, cambios o pagos, dime y te ayudo.

🤖 RipleyBot: 
🔍 Intencion detectada: seguimiento
¡Perfecto, Sebastian! Si compraste un producto por internet en Ripley, puedes hacer seguimiento de tu pedido ingresando a tu cuenta en ripley.cl. Allí encontrarás el estado de envío, tiempos estimados y opciones de retiro en tienda sin costo adicional.

Si necesitas cambiar el producto, recuerda que los cambios están sujetos a disponibilidad de stock.

¿Quieres saber el estado específico de tu pedido o necesitas ayuda con otro proceso? Si me das más detalles, te puedo orientar mejor.

🤖 RipleyBot: 
🔍 Intencion detectada: reclamo
Sebastian, lamento cualquier inconveniente que hayas tenido con tu compra y te pido disculpas por las molestias. Para realizar la d